### 1: Inisialisasi Environment & Library### 

In [3]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score
import time
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil dimuat.")

Library berhasil dimuat.


### 2: Memuat Dataset Bersih (Single Source of Truth)

In [4]:
# Memuat data hasil pembersihan dari Fase 1
# Pastikan path ini sesuai dengan struktur direktori VS Code Anda
file_path = '../dataset/processed/cleaned_binary_malicious_urls.csv'
df = pd.read_csv(file_path)

print(f"Dataset dimuat dengan {df.shape[0]} baris dan {df.shape[1]} kolom.")
# Melihat sekilas perbandingan kelas target (0: Benign, 1: Malicious)
print("\nDistribusi Kelas:")
print(df['label'].value_counts(normalize=True) * 100)

Dataset dimuat dengan 638564 baris dan 65 kolom.

Distribusi Kelas:
label
0    66.770598
1    33.229402
Name: proportion, dtype: float64


### 3: Pipeline Ekstraksi Fitur Leksikal (Domain Knowledge)

In [5]:
import urllib.parse
import re
import pandas as pd
import time

def extract_lexical_features(url):
    # 1. Penanganan Error urlparse (Fallback Mechanism)
    try:
        parsed = urllib.parse.urlparse(url)
        domain = parsed.netloc
        path = parsed.path
    except ValueError:
        # Jika URL sengaja dibuat cacat oleh penyerang (misal: Invalid IPv6)
        # Ekstrak domain menggunakan Regular Expression dasar
        domain_match = re.search(r'//([^/]+)', url)
        domain = domain_match.group(1) if domain_match else ""
        path = url.replace(domain, "") # Aproksimasi string sisa sebagai path
        
    # 2. Fitur Panjang & Rasio
    url_length = len(url)
    domain_length = len(domain)
    path_length = len(path)
    
    # Menghindari ZeroDivisionError
    digit_count = sum(c.isdigit() for c in url)
    lex_digit_ratio = digit_count / url_length if url_length > 0 else 0.0
    
    # 3. Fitur Kuantitas Karakter
    lex_qty_dot = url.count('.')
    lex_qty_hyphen = url.count('-')
    lex_qty_slash = url.count('/')
    lex_qty_question = url.count('?')
    lex_qty_equal = url.count('=')
    lex_qty_at = url.count('@')
    lex_qty_ampersand = url.count('&')
    
    # 4. Fitur Keberadaan Komponen (Domain Knowledge)
    # Deteksi penggunaan IP Address mentah di domain (IPv4)
    lex_use_ip = 1 if re.search(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', domain) else 0
    
    # Deteksi kata kunci rekayasa sosial
    security_keywords = ['login', 'verify', 'admin', 'bank', 'update', 'secure', 'account']
    lex_has_keyword = 1 if any(keyword in url.lower() for keyword in security_keywords) else 0
    
    # Kembalikan sebagai Pandas Series agar bisa di-unpack langsung ke DataFrame
    return pd.Series([
        url_length, domain_length, path_length, lex_qty_dot, lex_qty_hyphen, 
        lex_qty_slash, lex_qty_question, lex_qty_equal, lex_qty_at, 
        lex_qty_ampersand, lex_digit_ratio, lex_use_ip, lex_has_keyword
    ])

# ---------------------------------------------------------
# Cara Eksekusi (Tetap sama seperti kode Anda sebelumnya)
# ---------------------------------------------------------
start_time = time.time()

lexical_columns = [
    'url_length', 'domain_length', 'path_length', 'lex_qty_dot', 
    'lex_qty_hyphen', 'lex_qty_slash', 'lex_qty_question', 
    'lex_qty_equal', 'lex_qty_at', 'lex_qty_ampersand', 'lex_digit_ratio', 
    'lex_use_ip', 'lex_has_keyword'
]

df[lexical_columns] = df['url'].apply(extract_lexical_features)

print(f"Ekstraksi fitur leksikal selesai dalam {time.time() - start_time:.2f} detik.")

Ekstraksi fitur leksikal selesai dalam 33.87 detik.


### 4: Persiapan Ruang Pencarian (Search Space) GA

In [6]:
# Hapus kolom teks yang tidak diperlukan oleh model ML
# 'url' sudah diekstrak fiturnya, 'type' adalah metadata, 'domain' teks mentah
cols_to_drop = ['url', 'type', 'domain']
df_ml = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Pisahkan Fitur (X) dan Target (y)
X = df_ml.drop(columns=['label'])
y = df_ml['label']

print(f"Total ruang fitur awal untuk GA: {X.shape[1]} fitur.")

Total ruang fitur awal untuk GA: 73 fitur.


### 5: Formulasi Native Genetic Algorithm (GA) untuk Feature Selection

In [7]:
class GeneticAlgorithmFS:
    def __init__(self, estimator, cv, pop_size=20, generations=10, mutation_rate=0.05):
        self.estimator = estimator
        self.cv = cv
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.history_best_fitness = []
        self.best_chromosome = None
        
    def _initialize_population(self, n_features):
        # Inisialisasi gen (1 = aktif, 0 = tidak aktif) secara acak
        return np.random.randint(0, 2, size=(self.pop_size, n_features))
    
    def _calculate_fitness(self, population, X, y):
        fitness_scores = []
        f1_macro_scorer = make_scorer(f1_score, average='macro')
        
        for chromosome in population:
            # Jika semua gen 0, berikan fitness 0
            if np.sum(chromosome) == 0:
                fitness_scores.append(0)
                continue
                
            X_subset = X.iloc[:, chromosome == 1]
            # Menggunakan cross_val_score untuk evaluasi yang robust
            scores = cross_val_score(self.estimator, X_subset, y, cv=self.cv, scoring=f1_macro_scorer, n_jobs=-1)
            fitness_scores.append(scores.mean())
            
        return np.array(fitness_scores)
    
    def _crossover(self, parent1, parent2):
        # Single-point crossover
        crossover_point = np.random.randint(1, len(parent1)-1)
        child1 = np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])
        child2 = np.concatenate([parent2[:crossover_point], parent1[crossover_point:]])
        return child1, child2
    
    def _mutate(self, chromosome):
        # Bit-flip mutation
        for i in range(len(chromosome)):
            if np.random.rand() < self.mutation_rate:
                chromosome[i] = 1 - chromosome[i]
        return chromosome
    
    def fit(self, X, y):
        n_features = X.shape[1]
        population = self._initialize_population(n_features)
        
        for gen in range(self.generations):
            print(f"--- Generasi {gen + 1}/{self.generations} ---")
            fitness_scores = self._calculate_fitness(population, X, y)
            
            # Catat fitness terbaik di generasi ini
            best_idx = np.argmax(fitness_scores)
            self.history_best_fitness.append(fitness_scores[best_idx])
            self.best_chromosome = population[best_idx]
            
            print(f"Fitness Terbaik (F1-Macro): {fitness_scores[best_idx]:.4f} | Jumlah Fitur: {np.sum(self.best_chromosome)}")
            
            # Elitism: Pertahankan kromosom terbaik ke generasi berikutnya
            new_population = [self.best_chromosome]
            
            # Roulette Wheel / Tournament Selection (disederhanakan menggunakan propabilitas fitness)
            prob = fitness_scores / np.sum(fitness_scores)
            
            while len(new_population) < self.pop_size:
                # Pilih 2 orang tua
                p1_idx, p2_idx = np.random.choice(range(self.pop_size), size=2, p=prob, replace=True)
                child1, child2 = self._crossover(population[p1_idx], population[p2_idx])
                
                new_population.append(self._mutate(child1))
                if len(new_population) < self.pop_size:
                    new_population.append(self._mutate(child2))
                    
            population = np.array(new_population)
            
        return self

### 6: Eksekusi GA dan Ekspor Data yang Dioptimasi

In [9]:
# 1. Definisikan kolom yang BUKAN merupakan fitur (metadata & target)
# Sesuaikan 'timestamp_column' dengan nama kolom waktu yang ada di dataset Anda
columns_to_drop = ['url', 'type', 'label', 'timestamp_column'] 

# Alternatif yang lebih aman: Ambil HANYA kolom numerik
# df_numeric = df.select_dtypes(include=['int64', 'float64'])

# 2. Definisikan X (Fitur) dan y (Target)
X = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
y = df['label'] # Pastikan kolom ini sudah berbentuk 0 dan 1

# Pastikan X hanya berisi tipe data numerik untuk menghindari error terselubung
X = X.select_dtypes(include=['int64', 'float64'])

# 3. Pengambilan Sampel (Jika menggunakan X_sample dan y_sample)
# Misalnya Anda menggunakan 20% data untuk mempercepat evaluasi GA
from sklearn.model_selection import train_test_split
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.2, random_state=42, stratify=y)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

rf_estimator = RandomForestClassifier(random_state=42, n_jobs=-1)
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# 4. Inisialisasi dan Eksekusi GA (Sama seperti kode Anda)
import time

ga = GeneticAlgorithmFS(
    estimator=rf_estimator, 
    cv=cv_strategy, 
    pop_size=15, 
    generations=5, 
    mutation_rate=0.05
)

start_time = time.time()
print("Memulai evolusi Algoritma Genetika...")
ga.fit(X_sample, y_sample)
print(f"Evolusi selesai dalam {time.time() - start_time:.2f} detik.")

# Ekstraksi nama fitur terbaik
best_features_mask = ga.best_chromosome == 1
selected_features = X.columns[best_features_mask].tolist()

print(f"\nTotal fitur awal: {X.shape[1]}")
print(f"Total fitur terpilih optimal: {len(selected_features)}")
print(f"Daftar Fitur Terpilih:\n{selected_features}")

# ---------------------------------------------------------
# EKSPOR DATASET OPTIMAL
# ---------------------------------------------------------
# Buat dataset final yang HANYA berisi fitur terpilih + label
df_final_optimized = df_ml[selected_features + ['label']]

# Simpan untuk digunakan pada Notebook 03 (RF) dan 04 (XGBoost)
export_path = '../dataset/processed/ga_optimized_features_dataset.csv'
df_final_optimized.to_csv(export_path, index=False)
print(f"\nData berhasil diekspor ke: {export_path}")

Memulai evolusi Algoritma Genetika...
--- Generasi 1/5 ---
Fitness Terbaik (F1-Macro): 0.9229 | Jumlah Fitur: 46
--- Generasi 2/5 ---
Fitness Terbaik (F1-Macro): 0.9229 | Jumlah Fitur: 46
--- Generasi 3/5 ---
Fitness Terbaik (F1-Macro): 0.9229 | Jumlah Fitur: 46
--- Generasi 4/5 ---
Fitness Terbaik (F1-Macro): 0.9229 | Jumlah Fitur: 46
--- Generasi 5/5 ---
Fitness Terbaik (F1-Macro): 0.9229 | Jumlah Fitur: 46
Evolusi selesai dalam 300.73 detik.

Total fitur awal: 72
Total fitur terpilih optimal: 46
Daftar Fitur Terpilih:
['?', '=', '.', '%', '+', '!', '*', ',', '//', 'digits', 'letters', 'abnormal_url', 'https', 'Shortining_Service', 'having_ip_address', 'web_favicon', 'web_csp', 'web_xframe', 'web_password_fields', 'web_has_login', 'phish_urgency_words', 'phish_brand_hijack', 'phish_many_params', 'phish_suspicious_tld', 'phish_adv_exact_brand_match', 'phish_adv_brand_in_subdomain', 'phish_adv_hyphen_count', 'phish_adv_suspicious_tld', 'phish_adv_long_domain', 'phish_adv_many_subdomain